In [44]:
import argparse
import json
import tensorboard
import tensorboardX
import os
import re
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU, MutableLinear
from nni.nas.experiment.config import NasExperimentConfig
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from nni.experiment.config import utils, ExperimentConfig
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace, ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification

# Import the quantized CustomDARTSSpace
import sys
import importlib
import DoReFaLayers
importlib.reload(DoReFaLayers)
from DoReFaLayers import MutableDoReFaConv2d, MutableDoReFaLinear
from nni.nas.oneshot.pytorch.supermodule.operation import MixedOperation

def _safe_mixed_operation_freeze(self, sample):
    arguments = {**self.init_arguments}
    for name, mutable in self.mutable_arguments.items():
        arguments[name] = mutable.freeze(sample)
    operation = self.bound_type(**arguments)
    state_dict = self.freeze_weight(**arguments)
    operation.load_state_dict(state_dict, strict=False)
    return operation

MixedOperation.freeze = _safe_mixed_operation_freeze

# Quantized DARTS Architecture Search

This notebook performs NAS using the quantized CustomDARTSSpace with DoReFa quantization-aware training.

# Data Loading and Augmentation

In [34]:
def cutout_transform(img, length: int = 16):
    h, w = img.size(1), img.size(2)
    mask = np.ones((h, w), np.float32)
    y = np.random.randint(h)
    x = np.random.randint(w)

    y1 = np.clip(y - length // 2, 0, h)
    y2 = np.clip(y + length // 2, 0, h)
    x1 = np.clip(x - length // 2, 0, w)
    x2 = np.clip(x + length // 2, 0, w)

    mask[y1: y2, x1: x2] = 0.
    mask = torch.from_numpy(mask)
    mask = mask.expand_as(img)
    img *= mask
    return img

In [35]:
def get_cifar10_dataset(train: bool = True, cutout: bool = False):
    if train:
        transform_list = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(), 
        ]
        if cutout:
            transform_list.append(cutout_transform)
        transform = transforms.Compose(transform_list)
    else:
        transform = transforms.Compose([
            transforms.ToTensor(), 
        ])

    dataset = nni.trace(CIFAR10)(root='./data', train=train, download=True, transform=transform)
    
    return dataset

batch_size = 128
train_data = get_cifar10_dataset()
test_data = get_cifar10_dataset(train=False)

train_loader = DataLoader(
    train_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True, shuffle=True
)

valid_loader = DataLoader(
    test_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True
)

# Classification Module

In [36]:
@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__(self, learning_rate: float = 0.001, weight_decay: float = 0., auxiliary_loss_weight: float = 0.4, max_epochs: int = 600):
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay, export_onnx=False, num_classes=10)        
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=0.)
        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)        }

    def training_step(self, batch, batch_idx):
        """Training step, customized with auxiliary loss."""
        x, y = batch
        if self.auxiliary_loss_weight:
            y_hat, y_aux = self(x)
            loss_main = self.criterion(y_hat, y)
            loss_aux = self.criterion(y_aux, y)
            self.log('train_loss_main', loss_main)
            self.log('train_loss_aux', loss_aux)
            loss = loss_main + self.auxiliary_loss_weight * loss_aux
        else:
            y_hat = self(x)
            loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        return loss

    def on_train_epoch_start(self):
        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])

In [37]:
class CustomDARTSSpace(ModelSpace):
	"""Quantization-aware DARTS search space with DoReFa quantization.

	This corresponds to a flexible search space with:
	- LayerChoice on first 2 blocks (pool/conv ordering)
	- channel choices for layers 1-6
	- all learnable conv/linear ops quantized via DoReFa wrappers
	"""

	def __init__(
		self,
		input_channels: int = 3,
		channels: int = 64,
		num_classes: int = 10,
		layers: int = 7,
		verbose: int = 0,
		drop_path_prob: float = 0.1,
		num_bits: int = 4,
	):
		super().__init__()
		self.layers = nn.ModuleList()
		self.drop_path_prob = drop_path_prob
		self.verbose = verbose

		layer0_out = 16
		layer1_out = nni.choice("layer1_out_channels", [16, 32, 64])
		layer2_out = nni.choice("layer2_out_channels", [16, 32, 64])
		layer3_out = nni.choice("layer3_out_channels", [16, 32, 64])
		layer4_out = nni.choice("layer4_out_channels", [16, 32, 64])
		layer5_out = nni.choice("layer5_out_channels", [16, 32, 64])
		layer6_out = nni.choice("layer6_out_channels", [16, 32, 64])
		layer7_out = 22

		# Quantize first stem conv as well for full QAT consistency.
		self.preliminary_layer = MutableDoReFaConv2d(
			input_channels,
			layer0_out,
			kernel_size=3,
			padding=0,
			bias=False,
			num_bits=num_bits,
		)
		self.layer0_bn = MutableBatchNorm2d(layer0_out)
		self.layer0_relu = MutableReLU()

		layer1 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer0_out,
						layer1_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer1_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer0_out,
						layer1_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer1_out),
					MutableReLU(),
				),
			],
			label="layer_1",
		)
		self.layers.append(layer1)

		layer2 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer1_out,
						layer2_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer2_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer1_out,
						layer2_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer2_out),
					MutableReLU(),
				),
			],
			label="layer_2",
		)
		self.layers.append(layer2)

		layer3 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer2_out,
						layer3_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer3_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer2_out,
						layer3_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer3_out),
					MutableReLU(),
				),
			],
			label="layer_3",
		)
		self.layers.append(layer3)

		layer4 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer3_out,
						layer4_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer4_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer3_out,
						layer4_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer4_out),
					MutableReLU(),
				),
			],
			label="layer_4",
		)
		self.layers.append(layer4)

		layer5 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer4_out,
						layer5_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer5_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer4_out,
						layer5_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer5_out),
					MutableReLU(),
				),
			],
			label="layer_5",
		)
		self.layers.append(layer5)

		layer6 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer5_out,
						layer6_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer6_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer5_out,
						layer6_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer6_out),
					MutableReLU(),
				),
			],
			label="layer_6",
		)
		self.layers.append(layer6)

		layer7 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer6_out,
						layer7_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer7_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer6_out,
						layer7_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer7_out),
					MutableReLU(),
				),
			],
			label="layer_7",
		)
		self.layers.append(layer7)

		self.pool = nn.AdaptiveAvgPool2d((3, 3))
		feature1 = nni.choice("feature1", [32, 64, 128])
		feature2 = nni.choice("feature2", [32, 64, 128])
		feature3 = nni.choice("feature3", [32, 64])
		self.fc1 = MutableDoReFaLinear(198, feature1, num_bits=num_bits)
		self.fc2 = MutableDoReFaLinear(feature1, feature2, num_bits=num_bits)
		self.fc3 = MutableDoReFaLinear(feature2, feature3, num_bits=num_bits)
		self.relu = nn.ReLU()
		self.classifier = MutableDoReFaLinear(feature3, num_classes, num_bits=num_bits)

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		x = self.preliminary_layer(x)
		x = self.layer0_bn(x)
		x = self.layer0_relu(x)
		if self.verbose == 1:
			print(f"After preliminary layer: {x.shape}")

		for i, layer in enumerate(self.layers):
			x = layer(x)
			if self.verbose == 1:
				print(f"After layer {i + 1}: {x.shape}")
			if i in (1, 3, 6):
				x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
				if self.verbose == 1:
					print(f"After avg pooling: {x.shape}")

		x = self.pool(x)
		if self.verbose == 1:
			print(f"After adaptive pooling: {x.shape}")

		x = torch.flatten(x, 1)
		if self.verbose == 1:
			print(f"After flattening: {x.shape}")

		x = self.fc1(x)
		x = self.relu(x)
		if self.verbose == 1:
			print(f"After fc1: {x.shape}")

		x = self.fc2(x)
		x = self.relu(x)
		if self.verbose == 1:
			print(f"After fc2: {x.shape}")

		x = self.fc3(x)
		x = self.relu(x)
		if self.verbose == 1:
			print(f"After fc3: {x.shape}")

		x = self.classifier(x)
		if self.verbose == 1:
			print(f"After classifier: {x.shape}")

		return x

	def set_drop_path_prob(self, drop_path_prob: float):
		self.drop_path_prob = drop_path_prob
		for layer in self.layers:
			if hasattr(layer, "set_drop_path_prob"):
				layer.set_drop_path_prob(drop_path_prob)

# Evaluator Setup

Checkpoint callback and Lightning Evaluator configuration

In [38]:
# Checkpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='train_acc', 
    dirpath='./checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='max'
)

In [39]:
from nni.nas.evaluator.pytorch import Lightning, Trainer

max_epochs = 200

evaluator = Lightning(
    DartsClassificationModule(1e-2, 0., 0., max_epochs),
    Trainer(
        accelerator="auto",
        callbacks=[checkpoint_callback],
        max_epochs=max_epochs
    ),
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:139: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: <torch.utils.data.dataloader.DataLoader object at 0x7f1933619a10>
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:143: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: <torch.utils.data.dataloader.DataLoader object at 0x7f19332ddf10>


# NAS Search with Quantized Model

Using CustomDARTSSpace with DoReFa quantization

In [40]:
def get_next_experiment_name(experiment_working_directory: str, base_name: str = "PhotonicDARTS_Quant-v"):
    """Generate the next experiment name by incrementing the version number."""
    if not os.path.exists(experiment_working_directory):
        os.makedirs(experiment_working_directory)
    
    dirs = os.listdir(experiment_working_directory)
    pattern = re.compile(rf'{base_name}(\d+)')

    highest_i = 0
    for d in dirs:
        match = pattern.match(d)
        if match:
            i_value = int(match.group(1))
            if i_value > highest_i:
                highest_i = i_value
    return f"{base_name}{highest_i + 1}"

In [41]:
strategy = DartsStrategy(gradient_clip_val=0.)

def search(log_dir: str, batch_size: int = 128, num_bits: int = 4):
    """
    Run DARTS architecture search with quantized model.
    
    Parameters:
    -----------
    log_dir : str
        Directory for logging
    batch_size : int
        Batch size for training
    num_bits : int
        Number of bits for DoReFa quantization (default: 4)
    
    Returns:
    --------
    NasExperiment
        The completed NAS experiment with search results
    """
    
    # Define model search space with quantization
    model_space = CustomDARTSSpace(
        input_channels=3, 
        channels=64, 
        num_classes=10, 
        layers=7, 
        verbose=0,
        num_bits=num_bits  # Quantization parameter
    )
    model_space.set_drop_path_prob(0.)

    # Run NAS experiment
    exp_config = NasExperimentConfig.default(model_space, evaluator, strategy)
    exp_config.experiment_working_directory = "./DartsCheckpoints_Quant"
    exp_config.experiment_name = get_next_experiment_name("./DartsCheckpoints_Quant")
    exp_config.trial_concurrency = 1
    
    experiment = NasExperiment(model_space, evaluator, strategy, config=exp_config)
    experiment.run()

    return experiment

In [42]:
# Run the quantized DARTS search
# Set num_bits to 4 for 4-bit quantization (can adjust as needed)
experiment_results = search("./", 32, num_bits=4)

[2026-04-29 17:52:34] Config is not provided. Will try to infer.
[2026-04-29 17:52:34] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.


[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.


/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:139: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: {'train': <torch.utils.data.dataloader.DataLoader object at 0x7f1933619a10>, 'val': <torch.utils.data.dataloader.DataLoader object at 0x7f19332ddf10>}


[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: Checkpoint callback does not have last_model_path or best_model_path attribute. Either the strategy has not started, or it did not save any checkpoint: <pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint object at 0x7f19333d9250>
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] Checkpoint sav

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                      | Params | Mode  | FLOPs
------------------------------------------------------------------------------
0 | training_module | DartsClassificationModule | 465 K  | train | 0    
------------------------------------------------------------------------------
465 K     Trainable params
0         Non-trainable params
465 K     Total params
1.862     Total estimated model params size (MB)
167       Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 199: 100%|██████████████████| 391/391 [00:58<00:00,  6.64it/s, v_num=0, train_loss=0.254, train_acc=0.925]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199: 100%|██████████████████| 391/391 [00:58<00:00,  6.64it/s, v_num=0, train_loss=0.254, train_acc=0.925]
[2026-04-29 21:10:26] Waiting for models submitted to engine to finish...
[2026-04-29 21:10:26] Experiment is completed.
[2026-04-29 21:10:26] WARNING: `training_service` will be ignored for sequential execution engine.


# Results and Best Architecture

In [45]:
# Export best architecture
best_arch = experiment_results.export_top_models(formatter='instance', top_k=1)[0]
best_arch_desc = experiment_results.export_top_models(formatter='dict', top_k=1)[0]
best_arch_state_dict = best_arch.state_dict()

print("Best Architecture Description:")
print(best_arch_desc)

Best Architecture Description:
{'layer_1': 1, 'layer1_out_channels': 16, 'layer_2': 1, 'layer2_out_channels': 16, 'layer_3': 1, 'layer3_out_channels': 64, 'layer_4': 1, 'layer4_out_channels': 32, 'layer_5': 0, 'layer5_out_channels': 64, 'layer_6': 1, 'layer6_out_channels': 16, 'layer_7': 1, 'feature1': 32, 'feature2': 32, 'feature3': 32}


In [ ]:
# Experiment configuration
print("Experiment Configuration:")
print(experiment_results.config)

# Auto arch code generator

### Load Checkpoint

In [ ]:
checkpoint_path = './checkpoints/best-checkpoint-v51.ckpt'

checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

checkpoint_model = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=7, verbose=0, num_bits=4)

checkpoint_state_dict = checkpoint_model.state_dict()
pretrained_state_dict = checkpoint['state_dict']

if 'global_step' in checkpoint:
    print('Global Step:', checkpoint['global_step'])

if 'callbacks' in checkpoint and isinstance(checkpoint['callbacks'], dict):
    for key, callback in checkpoint['callbacks'].items():
        if isinstance(callback, dict) and 'best_model_score' in callback:
            print('Best Model Score (Train Accuracy):', callback['best_model_score'].item())

In [ ]:
experiment = NasExperiment(CustomDARTSSpace, evaluator, strategy)
experiment.load_checkpoint()
best_arch = experiment_results.export_top_models(formatter='instance', top_k=1)[0]
best_arch_desc = experiment_results.export_top_models(formatter='dict', top_k=1)[0]
best_arch_state_dict = best_arch.state_dict()

# From-Scratch Model

In [ ]:
class PhotonicArch(torch.nn.Module):
    def __init__(self, drop_path_prob=0.0, arch_dict=None):
        super().__init__()
        self.arch_dict = arch_dict
        self.drop_path_prob = drop_path_prob

        self.layer0_conv = torch.nn.Conv2d(3, 8, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(8)
        self.layer0_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_1'] == 0:
            self.layer1_conv = torch.nn.Conv2d(8, arch_dict['layer1_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer1_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer1_bn = torch.nn.BatchNorm2d(arch_dict['layer1_out_channels'], affine=True)
            self.layer1_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer1_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer1_conv = torch.nn.Conv2d(8, arch_dict['layer1_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer1_bn = torch.nn.BatchNorm2d(arch_dict['layer1_out_channels'], affine=True)
            self.layer1_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_2'] == 0:
            self.layer2_conv = torch.nn.Conv2d(arch_dict['layer1_out_channels'], arch_dict['layer2_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer2_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer2_bn = torch.nn.BatchNorm2d(arch_dict['layer2_out_channels'], affine=True)
            self.layer2_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer2_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer2_conv = torch.nn.Conv2d(arch_dict['layer1_out_channels'], arch_dict['layer2_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer2_bn = torch.nn.BatchNorm2d(arch_dict['layer2_out_channels'], affine=True)
            self.layer2_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_3'] == 0:
            self.layer3_conv = torch.nn.Conv2d(arch_dict['layer2_out_channels'], arch_dict['layer3_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer3_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer3_bn = torch.nn.BatchNorm2d(arch_dict['layer3_out_channels'], affine=True)
            self.layer3_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer3_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer3_conv = torch.nn.Conv2d(arch_dict['layer2_out_channels'], arch_dict['layer3_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer3_bn = torch.nn.BatchNorm2d(arch_dict['layer3_out_channels'], affine=True)
            self.layer3_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_4'] == 0:
            self.layer4_conv = torch.nn.Conv2d(arch_dict['layer3_out_channels'], arch_dict['layer4_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer4_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer4_bn = torch.nn.BatchNorm2d(arch_dict['layer4_out_channels'], affine=True)
            self.layer4_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer4_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer4_conv = torch.nn.Conv2d(arch_dict['layer3_out_channels'], arch_dict['layer4_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer4_bn = torch.nn.BatchNorm2d(arch_dict['layer4_out_channels'], affine=True)
            self.layer4_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_5'] == 0:
            self.layer5_conv = torch.nn.Conv2d(arch_dict['layer4_out_channels'], 22, kernel_size=3, stride=1, padding=1)
            self.layer5_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer5_bn = torch.nn.BatchNorm2d(22, affine=True)
            self.layer5_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer5_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer5_conv = torch.nn.Conv2d(arch_dict['layer4_out_channels'], 22, kernel_size=3, stride=1, padding=1)
            self.layer5_bn = torch.nn.BatchNorm2d(22, affine=True)
            self.layer5_relu = torch.nn.ReLU(inplace=False)

        self.pool = torch.nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = torch.nn.Linear(198, 160)
        self.fc2 = torch.nn.Linear(160, 128)
        self.fc3 = torch.nn.Linear(128, 96)
        self.relu = torch.nn.ReLU(inplace=False)
        self.classifier = torch.nn.Linear(96, 10)

    def forward(self, x):
        x = self.layer0_conv(x)
        x = self.layer0_bn(x)
        x = self.layer0_relu(x)

        x = self.layer1_avgpool(x)
        x = self.layer1_conv(x)
        x = self.layer1_bn(x)
        x = self.layer1_relu(x)

        x = self.layer2_conv(x)
        x = self.layer2_avgpool(x)
        x = self.layer2_bn(x)
        x = self.layer2_relu(x)

        x = self.layer3_conv(x)
        x = self.layer3_avgpool(x)
        x = self.layer3_bn(x)
        x = self.layer3_relu(x)

        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)

        x = self.layer4_avgpool(x)
        x = self.layer4_conv(x)
        x = self.layer4_bn(x)
        x = self.layer4_relu(x)

        x = self.layer5_avgpool(x)
        x = self.layer5_conv(x)
        x = self.layer5_bn(x)
        x = self.layer5_relu(x)

        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.classifier(x)
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob

In [ ]:
scratch_model = PhotonicArch(arch_dict=best_arch_desc)
scratch_model

### Rename dictionary keys

In [ ]:
best_arch_state_dict['layer0_conv.weight'] = best_arch_state_dict.pop('preliminary_layer.weight')
best_arch_state_dict['layer0_bn.weight'] = best_arch_state_dict.pop('layer0_bn.weight')
best_arch_state_dict['layer0_bn.bias'] = best_arch_state_dict.pop('layer0_bn.bias')
best_arch_state_dict['layer0_bn.running_mean'] = best_arch_state_dict.pop('layer0_bn.running_mean')
best_arch_state_dict['layer0_bn.running_var'] = best_arch_state_dict.pop('layer0_bn.running_var')
best_arch_state_dict['layer0_bn.num_batches_tracked'] = best_arch_state_dict.pop('layer0_bn.num_batches_tracked')

if 'layers.0.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer1_conv.weight'] = best_arch_state_dict.pop('layers.0.0.weight')
    best_arch_state_dict['layer1_conv.bias'] = best_arch_state_dict.pop('layers.0.0.bias')
else:
    best_arch_state_dict['layer1_conv.weight'] = best_arch_state_dict.pop('layers.0.1.weight')
    best_arch_state_dict['layer1_conv.bias'] = best_arch_state_dict.pop('layers.0.1.bias')
best_arch_state_dict['layer1_bn.weight'] = best_arch_state_dict.pop('layers.0.2.weight')
best_arch_state_dict['layer1_bn.bias'] = best_arch_state_dict.pop('layers.0.2.bias')
best_arch_state_dict['layer1_bn.running_mean'] = best_arch_state_dict.pop('layers.0.2.running_mean')
best_arch_state_dict['layer1_bn.running_var'] = best_arch_state_dict.pop('layers.0.2.running_var')
best_arch_state_dict['layer1_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.0.2.num_batches_tracked')

if 'layers.1.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer2_conv.weight'] = best_arch_state_dict.pop('layers.1.0.weight')
    best_arch_state_dict['layer2_conv.bias'] = best_arch_state_dict.pop('layers.1.0.bias')
else:
    best_arch_state_dict['layer2_conv.weight'] = best_arch_state_dict.pop('layers.1.1.weight')
    best_arch_state_dict['layer2_conv.bias'] = best_arch_state_dict.pop('layers.1.1.bias')
best_arch_state_dict['layer2_bn.weight'] = best_arch_state_dict.pop('layers.1.2.weight')
best_arch_state_dict['layer2_bn.bias'] = best_arch_state_dict.pop('layers.1.2.bias')
best_arch_state_dict['layer2_bn.running_mean'] = best_arch_state_dict.pop('layers.1.2.running_mean')
best_arch_state_dict['layer2_bn.running_var'] = best_arch_state_dict.pop('layers.1.2.running_var')
best_arch_state_dict['layer2_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.1.2.num_batches_tracked')

if 'layers.2.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer3_conv.weight'] = best_arch_state_dict.pop('layers.2.0.weight')
    best_arch_state_dict['layer3_conv.bias'] = best_arch_state_dict.pop('layers.2.0.bias')
else:
    best_arch_state_dict['layer3_conv.weight'] = best_arch_state_dict.pop('layers.2.1.weight')
    best_arch_state_dict['layer3_conv.bias'] = best_arch_state_dict.pop('layers.2.1.bias')
best_arch_state_dict['layer3_bn.weight'] = best_arch_state_dict.pop('layers.2.2.weight')
best_arch_state_dict['layer3_bn.bias'] = best_arch_state_dict.pop('layers.2.2.bias')
best_arch_state_dict['layer3_bn.running_mean'] = best_arch_state_dict.pop('layers.2.2.running_mean')
best_arch_state_dict['layer3_bn.running_var'] = best_arch_state_dict.pop('layers.2.2.running_var')
best_arch_state_dict['layer3_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.2.2.num_batches_tracked')

if 'layers.3.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer4_conv.weight'] = best_arch_state_dict.pop('layers.3.0.weight')
    best_arch_state_dict['layer4_conv.bias'] = best_arch_state_dict.pop('layers.3.0.bias')
else:
    best_arch_state_dict['layer4_conv.weight'] = best_arch_state_dict.pop('layers.3.1.weight')
    best_arch_state_dict['layer4_conv.bias'] = best_arch_state_dict.pop('layers.3.1.bias')
best_arch_state_dict['layer4_bn.weight'] = best_arch_state_dict.pop('layers.3.2.weight')
best_arch_state_dict['layer4_bn.bias'] = best_arch_state_dict.pop('layers.3.2.bias')
best_arch_state_dict['layer4_bn.running_mean'] = best_arch_state_dict.pop('layers.3.2.running_mean')
best_arch_state_dict['layer4_bn.running_var'] = best_arch_state_dict.pop('layers.3.2.running_var')
best_arch_state_dict['layer4_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.3.2.num_batches_tracked')

if 'layers.4.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer5_conv.weight'] = best_arch_state_dict.pop('layers.4.0.weight')
    best_arch_state_dict['layer5_conv.bias'] = best_arch_state_dict.pop('layers.4.0.bias')
else:
    best_arch_state_dict['layer5_conv.weight'] = best_arch_state_dict.pop('layers.4.1.weight')
    best_arch_state_dict['layer5_conv.bias'] = best_arch_state_dict.pop('layers.4.1.bias')
best_arch_state_dict['layer5_bn.weight'] = best_arch_state_dict.pop('layers.4.2.weight')
best_arch_state_dict['layer5_bn.bias'] = best_arch_state_dict.pop('layers.4.2.bias')
best_arch_state_dict['layer5_bn.running_mean'] = best_arch_state_dict.pop('layers.4.2.running_mean')
best_arch_state_dict['layer5_bn.running_var'] = best_arch_state_dict.pop('layers.4.2.running_var')
best_arch_state_dict['layer5_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.4.2.num_batches_tracked')

## Load weights

In [ ]:
scratch_model.load_state_dict(best_arch_state_dict)

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total_samples
    return avg_loss, accuracy

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

scratch_model.train()
module = DartsClassificationModule(1e-5, 0., 0., max_epochs)
module.set_model(scratch_model)
trainer = Trainer(max_epochs=module.max_epochs, precision=16, gradient_clip_val=0.1)

In [ ]:
trainer.fit(module, train_loader)

In [ ]:
criterion = nn.CrossEntropyLoss()
scratch_model.to(device)

avg_loss, accuracy = evaluate_model(scratch_model, valid_loader, criterion, device)
print(f'Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4%}')

In [ ]:
for inputs, labels in valid_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = scratch_model(inputs)
    print('Predicted:', outputs.argmax(dim=1))
    print('True:', labels)
    break